# Acoustic Topo-Lock | ASVspoof 2019 PA replay detection

This notebook uses the [public simulated PA release](https://doi.org/10.7488/ds/2555) for replay detection only. Read the [release and attribution terms](https://datashare.ed.ac.uk/items/31074a11-b6f6-4e92-a4ad-07093f8c0c45) before running. PA.zip is listed as 16.45 GB; the default free space check requires 19 GiB for a fresh download. Audio is read from the ZIP, so it is never extracted. You do not need the dataset on your computer.

Run the notebook once with `RUN_FULL = False`. Review the printed archive inspection and smoke report, then set `RUN_FULL = True` and rerun the configuration and full run cells. Smoke runs fit small train/dev subsets, skip evaluation and do not export a model. A CPU runtime is sufficient; full extraction and comparison may outlast a short Colab session.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import subprocess
import sys

# @title Runtime settings
REPO_URL = ""  # @param {type:"string"}
REPO_REF = ""  # @param {type:"string"}
RUN_FULL = False  # @param {type:"boolean"}
USE_DRIVE = False  # @param {type:"boolean"}

# For a private/unpublished checkout, leave REPO_URL blank and upload the source-only ZIP when prompted.
# For reproducibility on GitHub, set REPO_REF to the branch/tag, and retain the printed commit hash.
WORK = Path("/content")
SOURCE = WORK / "topolock-source"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = WORK / "drive/MyDrive/acoustic-topo-lock-pa"
else:
    DATA = WORK / "acoustic-topo-lock-pa"
ARCHIVE = DATA / "data" / "PA.zip"
OUTPUT = DATA / "results"
# Optional: point ARCHIVE to an existing authorized PA.zip in the cloud runtime.


In [ ]:
import io
from zipfile import ZipFile

MODULE = SOURCE / "replay_detection"
if not (MODULE / "pyproject.toml").is_file():
    if REPO_URL:
        if not REPO_URL.startswith("https://github.com/") or "@" in REPO_URL:
            raise ValueError("Enter a public GitHub HTTPS URL without credentials, or upload the source ZIP.")
        branch_args = ["--branch", REPO_REF] if REPO_REF else []
        subprocess.run(["git", "clone", "--depth", "1", *branch_args, "--", REPO_URL, str(SOURCE)], check=True)
    else:
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload only the replay detection source ZIP from this delivery.")
        with ZipFile(io.BytesIO(next(iter(uploaded.values())))) as bundle:
            entries = bundle.infolist()
            if sum(entry.file_size for entry in entries) > 8_000_000:
                raise ValueError("Use the source ZIP, not the dataset ZIP.")
            for entry in entries:
                target = (SOURCE / entry.filename).resolve()
                if not target.is_relative_to(MODULE.resolve()) or "\\" in entry.filename:
                    raise ValueError("Unexpected path in source ZIP.")
            bundle.extractall(SOURCE)

if not (MODULE / "pyproject.toml").is_file():
    raise FileNotFoundError("The checkout must contain replay_detection/pyproject.toml.")
if (SOURCE / ".git").exists():
    subprocess.run(["git", "-C", str(SOURCE), "rev-parse", "HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{MODULE}[test]"], check=True)
CONFIG = MODULE / "configs" / "replay_config.yaml"
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=MODULE, check=True)


The next cell downloads from Edinburgh DataShare only if the archive is absent. Interrupted transfers retain a `.part` file for retry when the server supports byte ranges. If the official host asks for access or terms acceptance, complete that step through the publisher and point `ARCHIVE` to the resulting authorized cloud copy. Review the discovered member names, protocol column counts, partition counts, label counts and key hashes before running training.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "replay_detection.dataset_loader", str(ARCHIVE), "--download"],
    cwd=MODULE, check=True,
)


In [ ]:
def run_training(mode):
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    run_dir = OUTPUT / f"{mode}-{stamp}"
    args = [sys.executable, "-m", "replay_detection.train",
            "--archive", str(ARCHIVE), "--config", str(CONFIG), "--out", str(run_dir)]
    if mode == "smoke":
        args.append("--smoke")
    subprocess.run(args, cwd=MODULE, check=True)
    return run_dir

smoke_dir = run_training("smoke")
smoke = json.loads((smoke_dir / "metrics.json").read_text())
print("Smoke run:", smoke_dir, "dev trials:", smoke["dev"]["trials"])


In [ ]:
if RUN_FULL:
    if "smoke_dir" not in globals() or not (smoke_dir / "metrics.json").is_file():
        raise RuntimeError("Run and review the smoke cell first.")
    run_dir = run_training("full")
    report = json.loads((run_dir / "metrics.json").read_text())
    print(json.dumps({k: report[k] for k in ("model_version", "selected_model", "dev", "eval",
                                             "inference_latency")}, indent=2))
else:
    print("Full run disabled. Review the smoke result before enabling RUN_FULL.")


In [ ]:
if RUN_FULL:
    from IPython.display import Image, display
    from replay_detection import load_replay_model

    detector = load_replay_model(run_dir)
    print(f"Verified {detector.model_version}: {run_dir}")
    shown_split = "eval" if report["eval"]["status"] == "measured" else "dev"
    for diagnostic in ("confusion", "roc", "det"):
        display(Image(filename=str(run_dir / f"{shown_split}_{diagnostic}.png")))
    print("Retain the run folder in trusted storage. Keep PA.zip, audio and audit logs out of GitHub.")
